# Module 04: Importance Methods Shootout

## Learning Objectives

By completing this notebook, you will:
1. Compare impurity-based (MDI), permutation-based (MDA), and SHAP importance on the same model with ground truth known
2. Quantify the bias in MDI due to high-cardinality and correlated features
3. Implement the knockoff filter procedure on a controlled dataset with exact FDR control
4. Measure ranking agreement between methods using Kendall's tau
5. Apply a practical recommendation framework for choosing an importance method

## Prerequisites
- Guide 03: Tree Importance and Knockoffs
- Notebook 02: Stability Selection

## Estimated Time: 15 minutes

---

## 1. Setup

We use LightGBM for speed and include SHAP (which has native LightGBM support). We construct two datasets:
- **Clean dataset**: independent features, known ground truth
- **Correlated dataset**: highly correlated features to expose bias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from scipy.stats import kendalltau, spearmanr
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP available.")
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available — install with: pip install shap")

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print("Setup complete.")

In [ ]:
# Dataset 1: Clean — independent features, known ground truth
# 5 true features (indices 0-4), 10 null features (indices 5-14)
# This is a regression problem to match Guide 01 and 02

def make_ground_truth_dataset(n=500, n_features=15, n_true=5, seed=42):
    """
    Create a regression dataset with independent features and known ground truth.

    Returns
    -------
    X_train, X_test, y_train, y_test : split arrays
    feature_names : list of feature name strings
    true_features : list of true feature indices
    true_coef : array of true coefficients
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, n_features))
    
    # True coefficients: decrease from 3.0 to 1.0 for the 5 true features
    true_coef = np.zeros(n_features)
    true_coef[:n_true] = np.linspace(3.0, 1.0, n_true)
    
    y = X @ true_coef + rng.normal(0, 0.5, n)
    
    feature_names = [f'feat_{i:02d}' for i in range(n_features)]
    X_df = pd.DataFrame(X, columns=feature_names)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_df, y, test_size=0.3, random_state=seed
    )
    return X_train, X_test, y_train, y_test, feature_names, list(range(n_true)), true_coef


X_train, X_test, y_train, y_test, FEATURE_NAMES, TRUE_IDX, TRUE_COEF = \
    make_ground_truth_dataset(n=500, n_features=15, n_true=5)

print(f"Dataset: {X_train.shape[0]} train, {X_test.shape[0]} test, {X_train.shape[1]} features")
print(f"True features: {TRUE_IDX}")
print(f"True coefficients: {TRUE_COEF[TRUE_IDX].round(2)}")

## 2. Train the Model and Compute All Three Importance Types

We train a single Random Forest and compute MDI, permutation importance, and SHAP on the same model. This isolates the effect of the importance method, holding the model constant.

In [ ]:
# Train Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

r2_val = rf.score(X_test, y_test)
print(f"Random Forest R² on test set: {r2_val:.4f}")

In [ ]:
# --- Importance Method 1: MDI (Mean Decrease in Impurity) ---
# Available directly from the fitted model — no extra computation needed
mdi_importance = rf.feature_importances_  # shape (n_features,)

# --- Importance Method 2: Permutation Importance ---
# Computed on the VALIDATION set (not training set — that would be misleading)
perm_result = permutation_importance(
    rf, X_test, y_test,
    n_repeats=30,       # repeat permutation 30 times to get stable estimates
    random_state=42,
    scoring='r2',
    n_jobs=-1
)
perm_importance = perm_result.importances_mean  # shape (n_features,)
perm_std = perm_result.importances_std

# --- Importance Method 3: SHAP ---
if SHAP_AVAILABLE:
    # TreeSHAP: exact computation for tree ensembles, O(TLD^2)
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)  # shape (n_test, n_features)
    # Global importance: mean |SHAP value| across all test instances
    shap_importance = np.abs(shap_values).mean(axis=0)  # shape (n_features,)
    print(f"SHAP computed: shape {shap_values.shape}")
else:
    # Fallback: use absolute mean prediction contribution (crude approximation)
    # In practice, always install SHAP for this type of analysis
    shap_importance = np.abs(perm_importance)  # approximation only
    print("Using permutation as SHAP fallback (install shap for real SHAP values)")

# Normalise all importances to [0,1] for fair visual comparison
def normalise(x):
    x = np.clip(x, 0, None)  # clip negative permutation importances to 0
    return x / max(x.max(), 1e-10)

mdi_norm = normalise(mdi_importance)
perm_norm = normalise(perm_importance)
shap_norm = normalise(shap_importance)

print("\nAll three importance scores computed and normalised.")

In [ ]:
# Side-by-side comparison plot
x = np.arange(len(FEATURE_NAMES))
width = 0.28
colors_bar = ['#e41a1c' if i in TRUE_IDX else '#aaaaaa' for i in range(len(FEATURE_NAMES))]

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

for ax, importance, label, color in [
    (axes[0], mdi_norm, 'MDI (Impurity)', '#d7191c'),
    (axes[1], perm_norm, 'Permutation Importance', '#2c7bb6'),
    (axes[2], shap_norm, 'SHAP Importance', '#1a9641'),
]:
    bar_colors = ['#e41a1c' if i in TRUE_IDX else '#aaaaaa' for i in range(len(FEATURE_NAMES))]
    bars = ax.bar(x, importance, color=bar_colors, width=0.7)
    ax.set_ylabel('Normalised Importance', fontsize=11)
    ax.set_title(f'{label}\n(Red = true features)', fontsize=12)
    ax.set_ylim(0, 1.1)
    
    # Annotate rank of each feature
    ranks = len(importance) - np.argsort(np.argsort(importance))
    for i, (imp, rank) in enumerate(zip(importance, ranks)):
        if imp > 0.05:
            ax.text(i, imp + 0.03, f'#{rank}', ha='center', fontsize=7)

axes[2].set_xticks(x)
axes[2].set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=9)

# Legend
true_patch = mpatches.Patch(color='#e41a1c', label='True feature')
null_patch = mpatches.Patch(color='#aaaaaa', label='Null feature')
fig.legend(handles=[true_patch, null_patch], loc='upper right', fontsize=10)

plt.suptitle('Feature Importance Shootout — Same Random Forest, Three Methods', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('importance_shootout.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Demonstrating MDI Bias with Correlated and High-Cardinality Features

We now construct a dataset designed to expose MDI's biases:
- A **high-cardinality null feature** (continuous random, independent of $y$)
- A **low-cardinality true feature** (binary, directly predictive of $y$)

MDI will rank the null continuous feature above the true binary feature. Permutation and SHAP will correctly rank the true feature first.

In [ ]:
# Bias demonstration dataset
rng = np.random.default_rng(99)
n_bias = 600

# True predictors
binary_true = rng.choice([0, 1], size=n_bias)    # binary, cardinality=2, TRUE
continuous_true = rng.normal(0, 1, n_bias)         # continuous, cardinality=inf, TRUE

# Null features
continuous_null = rng.normal(0, 1, n_bias)         # continuous, cardinality=inf, NULL
binary_null = rng.choice([0, 1], size=n_bias)     # binary, cardinality=2, NULL

# Three correlated features (each is a noisy version of continuous_true)
corr_a = 0.9 * continuous_true + np.sqrt(1 - 0.81) * rng.normal(0, 1, n_bias)  # TRUE (correlated)
corr_b = 0.9 * continuous_true + np.sqrt(1 - 0.81) * rng.normal(0, 1, n_bias)  # TRUE (correlated)

# Outcome: depends only on binary_true (strong effect) and continuous_true
y_bias = 3.0 * binary_true + 1.5 * continuous_true + 0.3 * rng.normal(0, 1, n_bias)

X_bias = pd.DataFrame({
    'binary_TRUE':     binary_true,      # cardinality=2, truly predictive
    'continuous_TRUE': continuous_true,  # cardinality=inf, truly predictive
    'corr_A_TRUE':     corr_a,          # correlated with continuous_TRUE
    'corr_B_TRUE':     corr_b,          # correlated with continuous_TRUE
    'continuous_NULL': continuous_null,  # cardinality=inf, null — MDI WILL OVERRANK THIS
    'binary_NULL':     binary_null,      # cardinality=2, null
})

TRUE_BIAS = ['binary_TRUE', 'continuous_TRUE', 'corr_A_TRUE', 'corr_B_TRUE']

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_bias, y_bias, test_size=0.3, random_state=42
)

# Train RF on bias dataset
rf_bias = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_bias.fit(X_b_train, y_b_train)

# Compute all three importance types
mdi_b = rf_bias.feature_importances_

perm_b = permutation_importance(
    rf_bias, X_b_test, y_b_test,
    n_repeats=30, random_state=42, scoring='r2', n_jobs=-1
).importances_mean

if SHAP_AVAILABLE:
    explainer_b = shap.TreeExplainer(rf_bias)
    shap_b = np.abs(explainer_b.shap_values(X_b_test)).mean(axis=0)
else:
    shap_b = normalise(perm_b)

print("Bias demonstration dataset trained.")
print(f"R² on test: {rf_bias.score(X_b_test, y_b_test):.4f}")

In [ ]:
# Plot bias demonstration
bias_features = list(X_bias.columns)
bar_colors_bias = ['#e41a1c' if f in TRUE_BIAS else '#aaaaaa' for f in bias_features]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, importance, label in [
    (axes[0], normalise(mdi_b), 'MDI\n(Biased)'),
    (axes[1], normalise(np.clip(perm_b, 0, None)), 'Permutation\n(Less biased)'),
    (axes[2], normalise(shap_b), 'SHAP\n(Unbiased)'),
]:
    ax.barh(bias_features, importance, color=bar_colors_bias[::-1][::-1])
    # Highlight the null continuous feature (MDI will overrank)
    for i, feat in enumerate(bias_features):
        if feat == 'continuous_NULL':
            ax.get_children()[i].set_edgecolor('black')
            ax.get_children()[i].set_linewidth(2)
    ax.set_xlabel('Normalised Importance', fontsize=10)
    ax.set_title(label, fontsize=11)
    ax.set_xlim(0, 1.15)

true_patch = mpatches.Patch(color='#e41a1c', label='True feature')
null_patch = mpatches.Patch(color='#aaaaaa', label='Null feature (boxed = continuous_NULL)')
fig.legend(handles=[true_patch, null_patch], loc='lower center', ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.1))

plt.suptitle('MDI Bias: continuous_NULL (boxed) is overranked by MDI\nbut correctly ranked near zero by Permutation and SHAP', fontsize=11)
plt.tight_layout()
plt.savefig('mdi_bias_demo.png', dpi=100, bbox_inches='tight')
plt.show()

# Rank of continuous_NULL under each method
null_idx = bias_features.index('continuous_NULL')
for label, imp in [('MDI', mdi_b), ('Permutation', perm_b), ('SHAP', shap_b)]:
    rank = len(imp) - np.argsort(np.argsort(imp))[null_idx]  # rank from top (1=most important)
    print(f"continuous_NULL rank under {label}: {rank}/{len(imp)}")

## 4. Knockoff Filter: FDR-Controlled Selection

We implement the second-order knockoff filter. For Gaussian features, we construct knockoff copies and compute the knockoff statistic $W_j = |Z_j| - |\tilde{Z}_j|$ where $Z_j$ and $\tilde{Z}_j$ are Lasso coefficients for the original and knockoff features respectively.

In [ ]:
from sklearn.linear_model import LassoCV

def construct_gaussian_knockoffs(X_arr):
    """
    Construct second-order Gaussian knockoff features.

    For X ~ N(mu, Sigma), knockoffs X_tilde satisfy:
      (X, X_tilde) ~ N([mu,mu], [[Sigma, Sigma-S],[Sigma-S, Sigma]])

    S is chosen as min(1, 2*lambda_min(Sigma)) * I — the equicorrelated knockoff.
    This is a simple closed-form approximation to the optimal SDP solution.

    Parameters
    ----------
    X_arr : array (n, p) — standardised features

    Returns
    -------
    X_knockoff : array (n, p) — knockoff features
    """
    n, p = X_arr.shape
    mu = X_arr.mean(axis=0)
    Sigma = np.cov(X_arr.T)  # (p, p)
    
    # Equicorrelated knockoff: S = s * I where s = min(2*lambda_min, 1)
    # This satisfies the positive semidefiniteness constraint 2*Sigma - S >= 0
    eigvals = np.linalg.eigvalsh(Sigma)
    lambda_min = max(eigvals.min(), 1e-6)  # numerical stability
    s = min(2 * lambda_min, 1.0)
    S_diag = s * np.ones(p)
    S = np.diag(S_diag)
    
    # Covariance of knockoff conditional on X:
    # Sigma_tilde|X = 2S - S * Sigma^{-1} * S
    Sigma_inv = np.linalg.inv(Sigma + 1e-6 * np.eye(p))  # regularised inverse
    Sigma_ko = 2 * S - S @ Sigma_inv @ S
    
    # Ensure positive semidefinite (numerical correction)
    eigvals_ko, eigvecs_ko = np.linalg.eigh(Sigma_ko)
    eigvals_ko = np.maximum(eigvals_ko, 0)  # clip negative eigenvalues
    Sigma_ko_psd = eigvecs_ko @ np.diag(eigvals_ko) @ eigvecs_ko.T
    
    # Mean of knockoff conditional on X:
    # mu_tilde|X = mu + (X - mu) @ (I - Sigma^{-1} @ S)
    mu_ko_given_X = mu + (X_arr - mu) @ (np.eye(p) - Sigma_inv @ S)
    
    # Sample knockoffs
    L = np.linalg.cholesky(Sigma_ko_psd + 1e-8 * np.eye(p))
    noise = np.random.default_rng(42).normal(0, 1, (n, p))
    X_knockoff = mu_ko_given_X + noise @ L.T
    
    return X_knockoff


# Use the clean ground-truth dataset from Section 2
X_arr = X_train.values
X_ko = construct_gaussian_knockoffs(X_arr)

# Check: knockoffs should have similar marginal distribution to originals
print("Knockoff construction check:")
print(f"  Original X mean: {X_arr.mean(axis=0)[:3].round(3)}")
print(f"  Knockoff  mean: {X_ko.mean(axis=0)[:3].round(3)}")
print(f"  Original X std:  {X_arr.std(axis=0)[:3].round(3)}")
print(f"  Knockoff  std:   {X_ko.std(axis=0)[:3].round(3)}")

In [ ]:
def knockoff_filter(X_orig, X_ko, y, fdr_level=0.1):
    """
    Apply the knockoff+ filter for FDR-controlled feature selection.

    Steps:
    1. Augment design matrix: [X_orig, X_knockoff]
    2. Fit Lasso on augmented matrix; get coefficients Z and Z_tilde
    3. Compute W_j = |Z_j| - |Z_tilde_j| (original beats knockoff = positive)
    4. Find threshold tau such that FDR <= fdr_level (knockoff+ procedure)
    5. Select features with W_j >= tau

    Parameters
    ----------
    X_orig : array (n, p)
    X_ko : array (n, p) — knockoff features
    y : array (n,)
    fdr_level : float — target FDR level (default 0.1 = 10%)

    Returns
    -------
    selected : array of int — selected feature indices
    W : array (p,) — knockoff statistics
    tau : float — selection threshold
    """
    n, p = X_orig.shape
    
    # Step 1: Augment design matrix
    X_aug = np.hstack([X_orig, X_ko])  # (n, 2p)
    
    # Step 2: Fit Lasso on augmented matrix
    # Use LassoCV to select lambda
    lasso_aug = LassoCV(cv=5, n_alphas=50, max_iter=5000, random_state=42)
    lasso_aug.fit(X_aug, y)
    
    # Split coefficients: first p are originals, last p are knockoffs
    coefs = lasso_aug.coef_
    Z = coefs[:p]    # original feature coefficients
    Z_tilde = coefs[p:]  # knockoff feature coefficients
    
    # Step 3: Compute knockoff statistic W_j = |Z_j| - |Z_tilde_j|
    # W_j > 0: original scored higher than knockoff (evidence of importance)
    # W_j < 0: knockoff scored higher (evidence against, acts as false positive estimate)
    W = np.abs(Z) - np.abs(Z_tilde)
    
    # Step 4: Knockoff+ threshold
    # Find smallest t > 0 such that FDP estimate <= fdr_level
    W_positive = W[W > 0]
    if len(W_positive) == 0:
        return np.array([]), W, np.inf
    
    t_values = np.sort(np.unique(np.abs(W[W != 0])))
    
    tau = np.inf
    for t in t_values:
        # Numerator: 1 + number of features with W_j <= -t (knockoff beat original)
        # This is a conservative estimate of false discoveries
        num_negative = np.sum(W <= -t)
        num_selected = np.sum(W >= t)
        
        if num_selected == 0:
            continue
        
        fdp_estimate = (1 + num_negative) / num_selected
        
        if fdp_estimate <= fdr_level:
            tau = t
            break
    
    # Step 5: Select features
    if np.isinf(tau):
        selected = np.array([], dtype=int)
    else:
        selected = np.where(W >= tau)[0]
    
    return selected, W, tau


# Run the knockoff filter at FDR=0.10 (10%)
print("Running knockoff filter (FDR level = 0.10)...")
selected_ko, W, tau = knockoff_filter(
    X_arr, X_ko, y_train.values, fdr_level=0.10
)

print(f"Knockoff threshold tau: {tau:.5f}")
print(f"Selected features: {selected_ko.tolist()}")
print(f"  -> {[FEATURE_NAMES[i] for i in selected_ko]}")
print(f"True features:     {TRUE_IDX}")
tp_ko = len(set(selected_ko) & set(TRUE_IDX))
fp_ko = len(set(selected_ko) - set(TRUE_IDX))
print(f"TP={tp_ko}, FP={fp_ko}, FDR_achieved = {fp_ko/max(len(selected_ko),1):.3f}")

In [ ]:
# Visualise knockoff statistics W_j
fig, ax = plt.subplots(figsize=(10, 4))

x_pos = np.arange(len(FEATURE_NAMES))
bar_colors_ko = ['#e41a1c' if i in TRUE_IDX else '#aaaaaa' for i in range(len(FEATURE_NAMES))]
bars = ax.bar(x_pos, W, color=bar_colors_ko, edgecolor='white')

# Mark selected features
if not np.isinf(tau):
    ax.axhline(tau, color='navy', linestyle='--', linewidth=2, label=f'Threshold τ={tau:.3f}')
    ax.axhline(-tau, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label=f'-τ (FDP numerator)')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('W_j = |Z_j| - |Z̃_j|', fontsize=12)
ax.set_title('Knockoff Statistics W_j\nFeatures above dashed line are selected (FDR ≤ 10%)', fontsize=13)
ax.legend(fontsize=10)

true_patch = mpatches.Patch(color='#e41a1c', label='True feature')
null_patch = mpatches.Patch(color='#aaaaaa', label='Null feature')
fig.legend(handles=[true_patch, null_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('knockoff_statistics.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Rank Agreement Analysis: Kendall's Tau Between Methods

Kendall's $\tau$ measures the fraction of concordant pairs in two rankings minus the fraction of discordant pairs. $\tau = 1$ means perfect agreement, $\tau = -1$ means perfect disagreement, $\tau = 0$ means no agreement.

We use the ground-truth ranking (by true coefficient magnitude) as the reference.

In [ ]:
# Compute Kendall's tau between each method's ranking and the ground truth ranking
# Ground truth ranking: by absolute true coefficient (TRUE_COEF)
ground_truth_rank = np.argsort(np.argsort(-np.abs(TRUE_COEF)))  # rank from 0 (best)

methods = {
    'MDI': mdi_importance,
    'Permutation': perm_importance,
    'SHAP': shap_importance,
}

print("Rank Agreement: Kendall's tau with ground truth ranking")
print("=" * 60)
print(f"{'Method':<20} {'Kendall tau':>15} {'p-value':>12}")
print("-" * 60)

tau_values = {}
for method_name, importance in methods.items():
    # Convert to ranking (lower rank = more important)
    method_rank = np.argsort(np.argsort(-importance))
    tau_val, p_val = kendalltau(ground_truth_rank, method_rank)
    tau_values[method_name] = tau_val
    print(f"{method_name:<20} {tau_val:>15.4f} {p_val:>12.4f}")

print("\nNote: Higher tau = better agreement with ground truth ranking.")
print("p-value < 0.05 means the agreement is statistically significant.")

# Pairwise Kendall's tau between methods
print("\nPairwise rank agreement between methods:")
method_names = list(methods.keys())
print(f"{'':>15}", end='')
for name in method_names:
    print(f"{name:>15}", end='')
print()
for name_i in method_names:
    print(f"{name_i:>15}", end='')
    for name_j in method_names:
        rank_i = np.argsort(np.argsort(-methods[name_i]))
        rank_j = np.argsort(np.argsort(-methods[name_j]))
        tau_ij, _ = kendalltau(rank_i, rank_j)
        print(f"{tau_ij:>15.3f}", end='')
    print()

In [ ]:
# Visualise the ranking comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (method_name, importance) in zip(axes, methods.items()):
    method_rank = np.argsort(np.argsort(-importance)) + 1  # 1-indexed ranks
    truth_rank = np.argsort(np.argsort(-np.abs(TRUE_COEF))) + 1
    
    colors_scatter = ['#e41a1c' if i in TRUE_IDX else '#aaaaaa' for i in range(len(FEATURE_NAMES))]
    ax.scatter(truth_rank, method_rank, c=colors_scatter, s=80, zorder=3)
    
    # Labels for true features
    for i in TRUE_IDX:
        ax.annotate(FEATURE_NAMES[i], (truth_rank[i], method_rank[i]),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')
    
    # Perfect agreement diagonal
    max_rank = len(FEATURE_NAMES)
    ax.plot([1, max_rank], [1, max_rank], 'k--', alpha=0.4, linewidth=1)
    
    tau_val = tau_values[method_name]
    ax.set_xlabel('True Rank (ground truth)', fontsize=10)
    ax.set_ylabel(f'{method_name} Rank', fontsize=10)
    ax.set_title(f'{method_name}\nKendall τ = {tau_val:.3f}', fontsize=11)

plt.suptitle('Rank Agreement: Method Rank vs Ground Truth Rank\n(Perfect agreement = diagonal line)', fontsize=12)
plt.tight_layout()
plt.savefig('rank_agreement.png', dpi=100, bbox_inches='tight')
plt.show()

## Exercise 5.1: Custom FDR Level for Knockoffs

**Task:** Run the knockoff filter at three FDR levels: 5%, 10%, and 20%. For each, report the number of features selected, true positives, and false positives.

**Expected result:** Higher FDR level → more features selected (higher recall, lower precision).

In [ ]:
# YOUR CODE HERE
# ---------------
# For fdr_level in [0.05, 0.10, 0.20]:
#   Run knockoff_filter(X_arr, X_ko, y_train.values, fdr_level=fdr_level)
#   Compute and print: n_selected, TP, FP, achieved FDR

fdr_levels = [0.05, 0.10, 0.20]

In [ ]:
# AUTO-GRADED TESTS — do not modify
def test_exercise_51_knockoff():
    fdr_levels = [0.05, 0.10, 0.20]
    n_selected_list = []
    for fdr in fdr_levels:
        sel, w, t = knockoff_filter(X_arr, X_ko, y_train.values, fdr_level=fdr)
        n_selected_list.append(len(sel))
        tp = len(set(sel.tolist()) & set(TRUE_IDX))
        fp = len(set(sel.tolist()) - set(TRUE_IDX))
        achieved_fdr = fp / max(len(sel), 1)
        print(f"FDR={fdr:.2f}: n_selected={len(sel)}, TP={tp}, FP={fp}, achieved_FDR={achieved_fdr:.3f}")
    
    # Monotonicity: higher FDR level should select >= features
    assert n_selected_list[0] <= n_selected_list[1] <= n_selected_list[2], \
        "Higher FDR level should select >= features (monotonicity violated)"
    print("Test passed: monotonicity holds across FDR levels.")

test_exercise_51_knockoff()

## Exercise 5.2: Importance Ranking for a New Dataset

**Task:** Use `make_classification` from sklearn to create a classification dataset with 20 features (10 informative, 10 redundant, noise=0). Train a Random Forest classifier, compute MDI and permutation importance, and compute Kendall's tau between the two rankings.

**Expected result:** The two rankings should agree more strongly than for the biased dataset from Section 3 (because `make_classification` uses independent informative features).

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Create dataset using make_classification
#    n_samples=400, n_features=20, n_informative=10, n_redundant=0, random_state=42

# 2. Train RandomForestClassifier

# 3. Compute mdi_clf = rf_clf.feature_importances_

# 4. Compute perm_clf using permutation_importance(..., scoring='accuracy')

# 5. Compute Kendall's tau between mdi_clf and perm_clf rankings

# 6. Print the tau value

In [ ]:
# AUTO-GRADED TESTS — do not modify
def test_exercise_52_ranking():
    from sklearn.datasets import make_classification
    from sklearn.ensemble import RandomForestClassifier
    
    X_clf, y_clf = make_classification(
        n_samples=400, n_features=20, n_informative=10,
        n_redundant=0, n_repeated=0, random_state=42
    )
    X_clf_df = pd.DataFrame(X_clf, columns=[f'f{i}' for i in range(20)])
    X_tr, X_te, y_tr, y_te = train_test_split(X_clf_df, y_clf, test_size=0.3, random_state=42)
    
    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_clf.fit(X_tr, y_tr)
    
    mdi_clf = rf_clf.feature_importances_
    perm_clf = permutation_importance(
        rf_clf, X_te, y_te, n_repeats=20, random_state=42, scoring='accuracy'
    ).importances_mean
    
    rank_mdi = np.argsort(np.argsort(-mdi_clf))
    rank_perm = np.argsort(np.argsort(-perm_clf))
    tau_val, p_val = kendalltau(rank_mdi, rank_perm)
    
    print(f"Kendall's tau (MDI vs Permutation): {tau_val:.4f} (p={p_val:.4f})")
    assert tau_val > 0.3, f"Expected tau > 0.3 for independent features, got {tau_val:.3f}"
    print("Test passed: MDI and permutation importance agree on independent features.")

test_exercise_52_ranking()

## 6. Practical Recommendation Framework

A decision flowchart for choosing the appropriate importance method based on dataset properties and goals.

In [ ]:
# Summary results table: all methods on the ground-truth dataset
print("Summary: All Methods on Ground-Truth Dataset (15 features, 5 true)")
print("=" * 75)
print(f"{'Method':<20} {'Top-5 features':<40} {'Tau vs truth':>12}")
print("-" * 75)

for method_name, importance in methods.items():
    top5 = [FEATURE_NAMES[i] for i in np.argsort(-importance)[:5]]
    rank_method = np.argsort(np.argsort(-importance))
    rank_truth = np.argsort(np.argsort(-np.abs(TRUE_COEF)))
    tau_val, _ = kendalltau(rank_truth, rank_method)
    print(f"{method_name:<20} {str(top5):<40} {tau_val:>12.4f}")

# Knockoff filter summary
print(f"{'Knockoff (FDR=10%)':<20} {str([FEATURE_NAMES[i] for i in selected_ko.tolist()]):<40} {'N/A (FDR ctrl)':>12}")

print("\nPractical Recommendations:")
print("  1. Quick exploration:          Use MDI — fast, available after training")
print("  2. Publication/reporting:      Use SHAP — theoretically grounded, instance-level")
print("  3. Correlated features:        Use Permutation or SHAP; avoid MDI")
print("  4. FDR control required:       Use Knockoff filter")
print("  5. Uncertainty over selection: Use Stability Selection (Notebook 02)")
print("  6. Deep learning models:       Use Attention importance + SHAP")

## Summary

### Key Takeaways

1. **MDI is biased** toward high-cardinality continuous features and toward features appearing higher in trees due to correlation. Always note these limitations when reporting MDI.

2. **Permutation importance** fixes the cardinality bias but underestimates importance for correlated features (the model compensates via correlated proxies). Compute on validation/test data, not training data.

3. **SHAP** is the theoretically principled method — consistent, additive, and handles interactions. TreeSHAP makes it efficient for tree ensembles. For correlated features, marginal SHAP values distribute attribution appropriately.

4. **The knockoff filter** is the only method with a provable FDR guarantee. It achieves this by constructing synthetic negative controls ($\tilde{X}_j$) and selecting features where the original score dominates the knockoff score.

5. **Kendall's tau** measures ranking agreement across methods. On independent features, MDI and permutation agree (tau > 0.5). On correlated or high-cardinality features, agreement drops and MDI diverges from ground truth.

### Module 04 Complete

You have now covered all three embedded method paradigms:
- **Regularisation paths** (Notebook 01): Lasso, ElasticNet, LARS
- **Stability selection** (Notebook 02): subsampling + FDR bound
- **Tree importance and knockoffs** (Notebook 03): MDI/MDA/SHAP comparison + FDR control

**Module 05** introduces genetic algorithms — a combinatorial approach that works with any black-box model and makes no convexity assumptions.

### Additional Resources
- [SHAP documentation](https://shap.readthedocs.io) — TreeSHAP, summary plots, interaction values
- Candès et al. (2018) "Model-X Knockoffs" — full theoretical treatment
- [knockpy package](https://github.com/amspector100/knockpy) — production knockoff implementation
- Strobl et al. (2007) "Bias in Variable Importance Measures" — MDI bias documentation